In [4]:
%env AWS_PROFILE=platform-developer

env: AWS_PROFILE=platform-developer


In [10]:
from utils.elasticsearch import get_client
from core.source import ElasticSource

es_client = get_client("read_only", "2025-10-02", "public")

source = ElasticSource(es_client=es_client, index_name="works-source-2025-10-02", query={"match_all": {}})

calm_source_works = {}

for work in source.stream_raw():
    source_id = work["state"]["sourceIdentifier"]
    if source_id["identifierType"]["id"] == "calm-record-id":
        calm_source_works[source_id["value"]] = work

2026-07-01 11:23:02 [info     ] Creating Elasticsearch client  es_mode=public host_config=https://0687c4947f404fe8ae10fa1aeab71bdc.eu-west-1.aws.found.io:443
2026-07-01 11:23:02 [info     ] Ran Elasticsearch query        duration_seconds=0 record_count=2000 slice_index=0
2026-07-01 11:23:03 [info     ] Ran Elasticsearch query        duration_seconds=0 record_count=2000 slice_index=4
2026-07-01 11:23:03 [info     ] Ran Elasticsearch query        duration_seconds=1 record_count=2000 slice_index=1
2026-07-01 11:23:03 [info     ] Ran Elasticsearch query        duration_seconds=1 record_count=2000 slice_index=0
2026-07-01 11:23:03 [info     ] Ran Elasticsearch query        duration_seconds=1 record_count=2000 slice_index=4
2026-07-01 11:23:03 [info     ] Ran Elasticsearch query        duration_seconds=1 record_count=2000 slice_index=3
2026-07-01 11:23:03 [info     ] Ran Elasticsearch query        duration_seconds=1 record_count=2000 slice_index=2
2026-07-01 11:23:04 [info     ] Ran Elastics

In [11]:
import polars as pl
import pickle

CALM_SOURCE_WORKS_PATH = "/Users/brychtas/Documents/work-snapshots/calm-source-works.parquet"

def save_parquet_snapshot(data: dict, path: str):
    df = pl.DataFrame({
        "id": list(data.keys()),
        "body": [pickle.dumps(v) for v in data.values()],
    })
    df.write_parquet(path)


save_parquet_snapshot(calm_source_works, CALM_SOURCE_WORKS_PATH)